# Zela: treino do modelo de Visão (YOLO11n + RDD2022)

1. Menu **Ambiente de execução → Alterar o tipo de ambiente → GPU T4**.
2. **Ambiente de execução → Executar tudo**. Leva cerca de 30 a 60 minutos.
3. No fim, o navegador baixa `zela_visao.zip` com o `best.pt`, as métricas e as imagens das transformações.
4. Coloque o `best.pt` em `vision/modelos/` no repositório e descompacte o resto em `vision/resultados/`.

Dataset: [RDD2022](https://doi.org/10.6084/m9.figshare.21431547.v1) (CC BY 4.0), partes Czech e China_MotorBike.

In [ ]:
!pip -q install ultralytics
import torch
assert torch.cuda.is_available(), "Sem GPU: troque o tipo de ambiente para GPU T4"
print(torch.cuda.get_device_name(0))

In [ ]:
%%writefile rdd_para_yolo.py
"""Baixa partes do RDD2022 e converte para o formato YOLO.

O RDD2022 é um único zip de 13 GB no figshare (CC BY 4.0), com um zip por
país dentro dele. Este script lê só os trechos dos países escolhidos usando
HTTP Range, sem baixar o arquivo inteiro.

Uso:
    python rdd_para_yolo.py --saida datasets/rdd --paises Czech China_MotorBike
"""

import argparse
import io
import random
import shutil
import urllib.request
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

URL_RDD2022 = "https://ndownloader.figshare.com/files/38030910"

# Classes que o Zela usa. As demais do RDD2022 (D43, D44, D50, Repair...) são ignoradas.
CLASSES = ["D00", "D10", "D20", "D40"]
NOMES = {
    "D00": "rachadura_longitudinal",
    "D10": "rachadura_transversal",
    "D20": "rachadura_jacare",
    "D40": "buraco",
}


class ArquivoHttp(io.RawIOBase):
    """Arquivo remoto que lê só os bytes pedidos (HTTP Range)."""

    def __init__(self, url: str):
        req = urllib.request.Request(url, headers={"Range": "bytes=0-0"})
        with urllib.request.urlopen(req) as r:
            self.url = r.url  # URL final, depois dos redirecionamentos
            self.tamanho = int(r.headers["Content-Range"].split("/")[1])
        self.pos = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, deslocamento, origem=0):
        base = {0: 0, 1: self.pos, 2: self.tamanho}[origem]
        self.pos = base + deslocamento
        return self.pos

    def readinto(self, buffer):
        if self.pos >= self.tamanho or len(buffer) == 0:
            return 0
        fim = min(self.pos + len(buffer), self.tamanho) - 1
        req = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{fim}"})
        with urllib.request.urlopen(req) as r:
            dados = r.read()
        buffer[: len(dados)] = dados
        self.pos += len(dados)
        return len(dados)


def baixar_paises(paises: list[str], pasta: Path) -> list[Path]:
    pasta.mkdir(parents=True, exist_ok=True)
    remoto = zipfile.ZipFile(io.BufferedReader(ArquivoHttp(URL_RDD2022), buffer_size=8 << 20))
    baixados = []
    for pais in paises:
        destino = pasta / f"{pais}.zip"
        if not destino.exists():
            membro = f"RDD2022/{pais}.zip"
            tamanho = remoto.getinfo(membro).compress_size / 1e6
            print(f"Baixando {membro} ({tamanho:.0f} MB)...")
            with remoto.open(membro) as origem, open(destino, "wb") as saida:
                shutil.copyfileobj(origem, saida, length=8 << 20)
        baixados.append(destino)
    return baixados


def voc_para_yolo(xml_path: Path, imagem: Path | None = None) -> list[str]:
    raiz = ET.parse(xml_path).getroot()
    largura = float(raiz.findtext("size/width") or 0)
    altura = float(raiz.findtext("size/height") or 0)
    if not largura or not altura:
        # Alguns XMLs vêm sem o tamanho; lemos da própria imagem.
        from PIL import Image

        with Image.open(imagem) as im:
            largura, altura = im.size
    linhas = []
    for obj in raiz.iter("object"):
        nome = obj.findtext("name")
        if nome not in CLASSES:
            continue
        caixa = obj.find("bndbox")
        x1, y1, x2, y2 = (float(caixa.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
        if x2 <= x1 or y2 <= y1:
            continue
        cx, cy = (x1 + x2) / 2 / largura, (y1 + y2) / 2 / altura
        w, h = (x2 - x1) / largura, (y2 - y1) / altura
        linhas.append(f"{CLASSES.index(nome)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return linhas


def montar_dataset(zips: list[Path], saida: Path, fracao_val: float, semente: int) -> Path:
    bruto = saida / "bruto"
    for z in zips:
        alvo = bruto / z.stem
        if not alvo.exists():
            print(f"Extraindo {z.name}...")
            with zipfile.ZipFile(z) as arquivo:
                arquivo.extractall(alvo)

    # Só a pasta train do RDD2022 tem anotações; o test oficial não tem.
    pares = []
    for xml in bruto.rglob("*.xml"):
        if "train" not in xml.parts:
            continue
        imagens = list(xml.parent.parent.parent.glob(f"images/{xml.stem}.jpg"))
        if imagens:
            pares.append((imagens[0], xml))
    if not pares:
        raise SystemExit("Nenhum par imagem + anotação encontrado. Confira a estrutura extraída.")

    random.Random(semente).shuffle(pares)
    n_val = int(len(pares) * fracao_val)
    divisoes = {"val": pares[:n_val], "train": pares[n_val:]}

    for divisao, itens in divisoes.items():
        (saida / "images" / divisao).mkdir(parents=True, exist_ok=True)
        (saida / "labels" / divisao).mkdir(parents=True, exist_ok=True)
        for imagem, xml in itens:
            shutil.copy(imagem, saida / "images" / divisao / imagem.name)
            rotulos = voc_para_yolo(xml, imagem)
            (saida / "labels" / divisao / f"{imagem.stem}.txt").write_text("\n".join(rotulos))
        print(f"{divisao}: {len(itens)} imagens")

    yaml = saida / "rdd.yaml"
    nomes = "\n".join(f"  {i}: {NOMES[c]}" for i, c in enumerate(CLASSES))
    yaml.write_text(f"path: {saida.resolve()}\ntrain: images/train\nval: images/val\nnames:\n{nomes}\n")
    return yaml


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--saida", default="datasets/rdd", type=Path)
    p.add_argument("--paises", nargs="+", default=["Czech", "China_MotorBike"])
    p.add_argument("--fracao-val", default=0.15, type=float)
    p.add_argument("--semente", default=42, type=int)
    args = p.parse_args()
    zips = baixar_paises(args.paises, args.saida / "zips")
    yaml = montar_dataset(zips, args.saida, args.fracao_val, args.semente)
    print(f"Dataset pronto: {yaml}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile transformacoes.py
"""Transformações intermediárias aplicadas a uma foto de denúncia.

A disciplina de Visão pede que o documento mostre cada etapa do
processamento. Este script salva uma imagem por etapa e um painel com todas.

Uso:
    python transformacoes.py foto.jpg --modelo best.pt --saida exemplos/
"""

import argparse
import json
from pathlib import Path

import cv2
import numpy as np

TAMANHO = 640


def letterbox(img: np.ndarray, tamanho: int = TAMANHO) -> np.ndarray:
    """Redimensiona mantendo a proporção e completa com cinza, como o YOLO faz."""
    h, w = img.shape[:2]
    escala = tamanho / max(h, w)
    nova = cv2.resize(img, (round(w * escala), round(h * escala)), interpolation=cv2.INTER_AREA)
    tela = np.full((tamanho, tamanho, 3), 114, dtype=np.uint8)
    y0, x0 = (tamanho - nova.shape[0]) // 2, (tamanho - nova.shape[1]) // 2
    tela[y0 : y0 + nova.shape[0], x0 : x0 + nova.shape[1]] = nova
    return tela


def etapas(img: np.ndarray) -> dict[str, np.ndarray]:
    """Cada etapa recebe a saída da anterior e devolve uma imagem para o relatório."""
    passos = {}
    passos["1_original"] = img
    redim = letterbox(img)
    passos["2_redimensionada_640"] = redim

    cinza = cv2.cvtColor(redim, cv2.COLOR_BGR2GRAY)
    passos["3_escala_de_cinza"] = cinza

    # CLAHE realça o contraste local: rachaduras em asfalto escuro aparecem melhor.
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(cinza)
    passos["4_contraste_clahe"] = clahe

    suave = cv2.GaussianBlur(clahe, (5, 5), 0)
    passos["5_desfoque_gaussiano"] = suave

    bordas = cv2.Canny(suave, 50, 150)
    passos["6_bordas_canny"] = bordas

    # Fechamento morfológico junta bordas próximas, formando o contorno do dano.
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    passos["7_morfologia_fechamento"] = cv2.morphologyEx(bordas, cv2.MORPH_CLOSE, kernel)
    return passos


def deteccoes(modelo_path: Path, img: np.ndarray):
    from ultralytics import YOLO

    resultado = YOLO(str(modelo_path)).predict(img, imgsz=TAMANHO, conf=0.25, verbose=False)[0]
    dados = [
        {
            "classe": resultado.names[int(c)],
            "confianca": round(float(p), 3),
            "caixa_xyxy": [round(float(v)) for v in xyxy],
        }
        for c, p, xyxy in zip(resultado.boxes.cls, resultado.boxes.conf, resultado.boxes.xyxy)
    ]
    return resultado.plot(), dados


def painel(passos: dict[str, np.ndarray], colunas: int = 4, lado: int = 320) -> np.ndarray:
    blocos = []
    for nome, im in passos.items():
        if im.ndim == 2:
            im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
        im = cv2.resize(letterbox(im), (lado, lado))
        cv2.putText(im, nome, (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        blocos.append(im)
    while len(blocos) % colunas:
        blocos.append(np.zeros_like(blocos[0]))
    linhas = [np.hstack(blocos[i : i + colunas]) for i in range(0, len(blocos), colunas)]
    return np.vstack(linhas)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("imagem", type=Path)
    p.add_argument("--modelo", type=Path)
    p.add_argument("--saida", type=Path, default=Path("exemplos"))
    args = p.parse_args()

    img = cv2.imread(str(args.imagem))
    if img is None:
        raise SystemExit(f"Não consegui abrir {args.imagem}")
    passos = etapas(img)
    if args.modelo:
        anotada, dados = deteccoes(args.modelo, img)
        passos["8_deteccao_yolo"] = anotada
    else:
        dados = []

    pasta = args.saida / args.imagem.stem
    pasta.mkdir(parents=True, exist_ok=True)
    for nome, im in passos.items():
        cv2.imwrite(str(pasta / f"{nome}.jpg"), im)
    cv2.imwrite(str(pasta / "painel.jpg"), painel(passos))
    (pasta / "deteccoes.json").write_text(json.dumps(dados, indent=2, ensure_ascii=False))
    print(f"{len(passos)} etapas salvas em {pasta}; {len(dados)} danos detectados")


if __name__ == "__main__":
    main()

## 1. Baixar e converter o dataset (cerca de 450 MB)

In [ ]:
!python rdd_para_yolo.py --saida /content/datasets/rdd --paises Czech China_MotorBike

## 2. Treinar

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolo11n.pt")  # pesos pré-treinados no COCO
modelo.train(
    data="/content/datasets/rdd/rdd.yaml",
    epochs=60,
    imgsz=640,
    batch=32,
    patience=15,  # para cedo se a validação parar de melhorar
    seed=42,
    project="/content/runs",
    name="zela",
    exist_ok=True,
)

## 3. Avaliar

In [ ]:
import json
from pathlib import Path

melhor = YOLO("/content/runs/zela/weights/best.pt")
m = melhor.val(data="/content/datasets/rdd/rdd.yaml", imgsz=640, split="val")
metricas = {
    "mAP50": round(float(m.box.map50), 4),
    "mAP50-95": round(float(m.box.map), 4),
    "precisao": round(float(m.box.mp), 4),
    "recall": round(float(m.box.mr), 4),
    "mAP50_por_classe": {melhor.names[i]: round(float(v), 4) for i, v in enumerate(m.box.ap50)},
}
print(json.dumps(metricas, indent=2, ensure_ascii=False))
Path("/content/entrega").mkdir(exist_ok=True)
Path("/content/entrega/metricas.json").write_text(json.dumps(metricas, indent=2, ensure_ascii=False))

## 4. Transformações intermediárias em 3 imagens de validação

In [ ]:
import random
imagens = sorted(Path("/content/datasets/rdd/images/val").glob("*.jpg"))
for img in random.Random(7).sample(imagens, 3):
    !python transformacoes.py {img} --modelo /content/runs/zela/weights/best.pt --saida /content/entrega/exemplos

## 5. Empacotar e baixar

In [ ]:
import shutil
from google.colab import files

for nome in ["weights/best.pt", "results.png", "confusion_matrix.png", "BoxPR_curve.png"]:
    origem = Path("/content/runs/zela") / nome
    if origem.exists():
        shutil.copy(origem, "/content/entrega/")
shutil.make_archive("/content/zela_visao", "zip", "/content/entrega")
files.download("/content/zela_visao.zip")